# AMASE-P Scheduling — Interactive Demo

Step-by-step walkthrough of the standard workflow (the notebook twin of `example/demo.py`):

1. **Load** the target list (with input validation)
2. **Precompute** the visibility cache in parallel (once)
3. **Single-night** schedule (reuses the cache)
4. **Multi-night campaign** 2027-04-01 .. 2027-04-15 with a weather model
5. **Export** the observing log (text + CSV trio) and all figures

Everything is written to `example/outputs/`. The same results can be produced from the CLI;
the equivalent commands are shown at the end.

In [ ]:
from pathlib import Path
import time

import pandas as pd
from IPython.display import Image, display

from amase_scheduling import (
    Scheduler,
    VisibilityCache,
    format_report,
    load_targets,
    plot_campaign_figure,
    plot_night_figure,
    save_all_night_figures,
    save_nights_csv,
    save_schedule_csv,
    save_targets_csv,
)

# Locate the repo root (works whether the kernel starts in the repo or in example/)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "example" / "targets.csv").exists())
TARGETS = REPO / "example" / "targets.csv"
OUTDIR = REPO / "example" / "outputs"
OUTDIR.mkdir(parents=True, exist_ok=True)

START, END = "2027-04-01", "2027-04-15"
CLEAR_PROB = 0.5   # per-night clear probability for the campaign simulation
SEED = 2027        # weather RNG seed -> fully reproducible
WORKERS = 8        # parallel workers for the visibility precompute

print(f"repo: {REPO}\noutputs: {OUTDIR}")

## Step 1 — Load the target list

`load_targets` parses the CSV and validates every row (missing fields, coordinate forms,
`exp_time <= 3600`, `n_dither` in {1, 3, 9, 27}, `n_exposure >= 1`; a missing `group` becomes
`"Untitle"`). Each target's **block** = `exp_time x n_dither` (indivisible); `n_exposure` is the
**season-total** visit demand carried across nights.

In [ ]:
targets = load_targets(str(TARGETS))

overview = pd.DataFrame(
    {
        "name": t.name,
        "ra_deg": round(t.coord.ra.deg, 4),
        "dec_deg": round(t.coord.dec.deg, 4),
        "priority": t.priority,
        "block_min": round(t.block_duration_sec / 60, 1),
        "n_exposure": t.n_exposure,
        "total_h": round(t.total_time_sec / 3600, 2),
        "group": t.group,
    }
    for t in targets
)
print(f"{len(targets)} targets, {overview['n_exposure'].sum()} visits, "
      f"{overview['total_h'].sum():.1f} h total demand, {overview['group'].nunique()} groups")
overview

## Step 2 — Precompute the visibility cache

Visibility (altitude / Sun / Moon per 5-min slot, valid block-start slots, transit-quality
factor) is independent of weather and scheduling state, so it can be computed **once, in
parallel across nights**, and reused by any number of scheduling runs. The cache is bound to
the target list (names + order) and the date range by a fingerprint — a mismatch raises a
clear error at scheduling time.

In [ ]:
t0 = time.perf_counter()
cache = VisibilityCache.build(targets, START, END, n_workers=WORKERS)
cache.save(str(OUTDIR / "vis_cache.npz"))
print(f"cached {len(cache)} nights in {time.perf_counter() - t0:.1f}s -> {OUTDIR / 'vis_cache.npz'}")

## Step 3 — Single-night schedule

A single night is a degenerate campaign: one date, `clear_prob = 1.0` (no weather dice), full
visit demand. The same engine and the same unified report serve both cases — for one night
the night section expands into the full block table.

In [ ]:
scheduler = Scheduler()
night_result = scheduler.schedule(targets, START, visibility_cache=cache)
print(format_report(night_result))

In [ ]:
night_png = OUTDIR / f"night_{START}.png"
plot_night_figure(night_result.nights[0], targets, scheduler.observer, str(night_png))
display(Image(filename=str(night_png)))

## Step 4 — Multi-night campaign with weather

Each night is independently clear with probability `clear_prob` (Bernoulli, seeded). Scheduled
visits decrement each target's `remaining` counter; completed targets graduate. The engine is
serial across nights — per-night MILPs are exact here because blocks never span nights and no
inter-night constraints exist (see DESIGN.md §8.4).

In [ ]:
result = scheduler.schedule(
    targets, START, END,
    clear_prob=CLEAR_PROB, seed=SEED,
    visibility_cache=cache, verbose=True,
)
report = format_report(result)
print(report)

In [ ]:
campaign_png = OUTDIR / "campaign.png"
plot_campaign_figure(result, str(campaign_png), targets=targets, observer=scheduler.observer)
display(Image(filename=str(campaign_png)))

## Step 5 — Export the observing log and figures

The CSV trio (`observing_log.csv` + `_targets.csv` + `_nights.csv`) is the complete interchange
format — `amase-plot` can re-render every figure from these files alone, without re-running
the scheduler.

In [ ]:
(OUTDIR / "observing_log.txt").write_text(report + "\n")
save_schedule_csv(result, str(OUTDIR / "observing_log.csv"))
save_targets_csv(result, str(OUTDIR / "observing_log_targets.csv"))
save_nights_csv(result, str(OUTDIR / "observing_log_nights.csv"))
night_paths = save_all_night_figures(result, targets, scheduler.observer, str(OUTDIR / "nights"))
print(f"CSV trio + report + {len(night_paths)} night figures written to {OUTDIR}")
pd.read_csv(OUTDIR / "observing_log.csv").head(10)

In [ ]:
# One of the per-night duty figures
display(Image(filename=night_paths[3]))

## Equivalent CLI session

```bash
amase-precompute example/targets.csv --start 2027-04-01 --end 2027-04-15 \
    --workers 8 -o example/outputs/vis_cache.npz

amase-schedule example/targets.csv 2027-04-01 --cache example/outputs/vis_cache.npz \
    -o example/outputs/night.csv

amase-schedule example/targets.csv --start 2027-04-01 --end 2027-04-15 \
    --clear-prob 0.5 --seed 2027 --cache example/outputs/vis_cache.npz \
    -o example/outputs/observing_log.csv -v

amase-plot campaign example/outputs/observing_log.csv --targets example/targets.csv \
    -o example/outputs/campaign.png
amase-plot nights example/outputs/observing_log.csv --targets example/targets.csv \
    -o example/outputs/nights/
```